# Forecast baseline y punto de reorden

1. Clasificación ABC por contribución de ventas
2. Features rolling de demanda por SKU
3. Backtest temporal baseline (MAE/MAPE)
4. Tabla de reorden por producto

La demanda diaria usada en `train` / `predict` aplica higiene (`prepare_daily_demand`): ver notebook 03.

In [1]:
from pathlib import Path
import sys

import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "inventario_ecommerce").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from inventario_ecommerce import config
from inventario_ecommerce.dataset import load_transactions, save_processed
from inventario_ecommerce.features import (
    clean_transactions,
    sales_by_product_last_quarter,
    compute_abc_classification,
    build_daily_sku_demand,
    build_rolling_features,
)
from inventario_ecommerce.modeling.train import temporal_backtest_baseline
from inventario_ecommerce.modeling.predict import forecast_30d_baseline, build_reorder_policy

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
raw = load_transactions()
clean = clean_transactions(raw)
daily = build_daily_sku_demand(clean)
rolling = build_rolling_features(daily)

latest_features = (
    rolling.sort_values("Date")
    .groupby([config.COL_STOCK_CODE, config.COL_DESCRIPTION], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

print(f"Raw rows: {len(raw):,}")
print(f"Clean rows: {len(clean):,}")
print(f"Daily rows: {len(daily):,}")
print(f"SKUs: {latest_features[config.COL_STOCK_CODE].nunique():,}")

Raw rows: 1,067,371
Clean rows: 1,037,339
Daily rows: 532,929
SKUs: 4,900


In [3]:
last_q = sales_by_product_last_quarter(clean)
abc = compute_abc_classification(last_q)
abc["ABCClass"].value_counts()

ABCClass
C    1899
B     803
A     684
Name: count, dtype: int64

In [4]:
sku_metrics, global_metrics = temporal_backtest_baseline(daily, horizon_days=30, lookback_days=30)
global_metrics

,CutoffDate,EvalStartDate,EvalEndDate,GlobalMAE,GlobalMAPE,HorizonDays,LookbackDays
0,2011-11-09,2011-11-10,2011-12-09,20.19,283.78,30,30


In [5]:
forecast = forecast_30d_baseline(daily, lookback_days=30, horizon_days=30)
policy = build_reorder_policy(latest_features, forecast, abc)
policy.head(20)

,StockCode,Description,ABCClass,TotalSales,forecast_daily,forecast_30d,demand_mean_30d,demand_std_30d,demand_cv_30d,lead_time_demand,safety_stock,reorder_point,target_stock,recommended_order_qty,recommendation
0,23843,"PAPER CRAFT , LITTLE BIRDIE",A,"168,469.60","80,995.00","2,429,850.00","80,995.00",0.00,0.00,"1,133,930.00",0.00,"1,133,930.00","1,700,895.00","566,965.00",Monitoreo diario; evitar quiebres
1,84826,ASSTD DESIGN 3D PAPER STICKERS,C,36.18,"4,211.00","126,330.00",463.27,"2,281.56",4.92,"58,954.00","10,927.10","69,881.10","99,358.10","29,477.00",Revisión quincenal
2,23084,RABBIT NIGHT LIGHT,A,"56,894.39",586.27,"17,588.08",576.13,543.61,0.94,"8,207.77","3,823.96","12,031.73","16,135.61","4,103.88",Monitoreo diario; evitar quiebres
3,22197,POPCORN HOLDER,A,"27,002.40",556.46,"16,693.85",600.83,679.82,1.13,"7,790.46","4,782.08","12,572.54","16,467.77","3,895.23",Monitoreo diario; evitar quiebres
4,21232,STRAWBERRY CERAMIC TRINKET BOX,A,"3,062.15",364.00,"10,920.00",64.60,105.57,1.63,"5,096.00",742.63,"5,838.63","8,386.63","2,548.00",Monitoreo diario; evitar quiebres
5,22086,PAPER CHAIN KIT 50'S CHRISTMAS,A,"50,907.49",284.65,"8,539.62",281.27,171.39,0.61,"3,985.15","1,205.59","5,190.74","7,183.32","1,992.58",Monitoreo diario; evitar quiebres
6,21051,found,C,0.00,240.00,"7,200.00",240.00,0.00,0.00,"3,360.00",0.00,"3,360.00","5,040.00","1,680.00",Revisión quincenal
7,21915,John Lewis,C,0.00,200.00,"6,000.00",200.00,0.00,0.00,"2,800.00",0.00,"2,800.00","4,200.00","1,400.00",Revisión quincenal
8,85099B,JUMBO BAG RED RETROSPOT,A,"31,101.76",193.31,"5,799.23",205.87,240.87,1.17,"2,706.31","1,694.39","4,400.70","5,753.85","1,353.15",Monitoreo diario; evitar quiebres
9,22578,WOODEN STAR CHRISTMAS SCANDINAVIAN,A,"3,596.04",192.35,"5,770.38",198.80,147.52,0.74,"2,692.85","1,037.74","3,730.58","5,077.01","1,346.42",Monitoreo diario; evitar quiebres


In [6]:
save_processed(abc, "abc_last_quarter.csv")
save_processed(latest_features, "sku_rolling_features_latest.csv")
save_processed(sku_metrics, "forecast_backtest_by_sku.csv")
save_processed(global_metrics, "forecast_backtest_global.csv")
save_processed(policy, "inventory_reorder_recommendations.csv")

print("Artefactos guardados en:", config.PROCESSED_DATA_DIR)

Artefactos guardados en: C:\code\proyecto-a-inventario-ecommerce\data\processed


## Salida

Tabla: `data/processed/inventory_reorder_recommendations.csv`

Campos clave: `ABCClass`, `forecast_30d`, `safety_stock`, `reorder_point`, `target_stock`.